One-Class SVM

Одноклассовая машина опорных векторов (SVM) — это разновидность традиционной SVM. Она специально разработана для выявления аномалий. 

Ее основная задача — находить примеры, которые заметно отличаются от нормы. В отличие от традиционных моделей машинного обучения, ориентированных на бинарную или многоклассовую классификацию, одноклассовая SVM специализируется на выявлении выбросов или новых элементов в наборах данных.

In [65]:
import pandas as pd
import numpy as np

url = "https://huggingface.co/datasets/vansh11/Network_Anomaly_Detection/resolve/main/Train.txt"
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack', 'last_flag'
]

df = pd.read_csv(url, header=None, names=columns)

df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,last_flag
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  object 
 2   service                      125973 non-null  object 
 3   flag                         125973 non-null  object 
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13 

Это датасет сетевого трафика

last_flag - служебный столбец, он соответствует уровню сложности записи;


attack - по сути это целевая переменная, то есть эти данные нужны для классификации.



In [67]:
df['attack'].unique()

array(['normal', 'neptune', 'warezclient', 'ipsweep', 'portsweep',
       'teardrop', 'nmap', 'satan', 'smurf', 'pod', 'back',
       'guess_passwd', 'ftp_write', 'multihop', 'rootkit',
       'buffer_overflow', 'imap', 'warezmaster', 'phf', 'land',
       'loadmodule', 'spy', 'perl'], dtype=object)

normal - нормальный трафик.

всё остальное - разные типы атак

Создадим флажок

In [68]:
df['true_anomaly'] = df['attack'].apply(lambda x: 0 if x == 'normal' else 1)

In [69]:
from scipy.optimize import minimize
class OneClassSVM:
    def __init__(self, nu=0.1, gamma=None, tol=1e-6, max_iter=1000):
        self.nu = nu
        self.gamma = gamma
        self.tol = tol
        self.max_iter = max_iter


        self.X_train = None
        self.alpha = None
        self.rho = None
        self.C = None
        self.support_vectors_ = None
        self.support_indices_ = None
    
    def _rbf_kernel(self, A, B):             # Используем RBF-ядро
        A_sq = np.sum(A ** 2, axis=1).reshape(-1, 1)
        B_sq = np.sum(B ** 2, axis = 1).reshape(1, -1)
        distances = A_sq + B_sq - 2 * A @ B.T       # По сути, это матрица расстояний
        return np.exp(-self.gamma * distances)
    
    def fit(self, X):
        X = np.array(X, dtype=float)
        self.X_train = X
        n_samples, n_features = X.shape
        if self.gamma is None:
            self.gamma = 1 / n_features

        self.C = 1 / (self.nu * n_samples)
        K = self._rbf_kernel(X, X)        # Сравниваем объекты друг с другом. По диагонали будут значения K(xi, xj) = 1

        # Дальше идет оптимизация
        def objective(alpha):
            return 0.5 * alpha @ K @ alpha             # Это функция, которую мы минимизируем: 1/2 * alpha.T * K * alpha
    
        def grad(alpha):
            return alpha @ K                  # производная этой функции. Так оптимизатор быстрее найдет максимум
    
        constraints = {
            'type': 'eq',          # Это значит равенство!!! equal = РАВЕНСТВО
            'fun': lambda alpha: np.sum(alpha) - 1, # Ограничение: общая сумма альф = 1
            'jac': lambda alpha: np.ones_like(alpha)    #Производная ограничения. производная по каждому альфа равна 1. (dg / dalpha = 1)
            }
        bounds = [(0, self.C) for i in range(n_samples)]    # Это ограничение: 0 <= alpha <= C для каждого объекта

        alpha0 = np.ones(n_samples) / n_samples

        result = minimize(
            fun=objective,
            x0=alpha0,
            jac=grad,
            bounds=bounds,
            constraints=constraints,
            method='SLSQP',     # Метод оптимизации, который умеет работать с ограничениями
            options={'maxiter': self.max_iter,
                'ftol': self.tol}   #Точность остановки
                            )
        self.alpha = result.x      # result.X - результат работы оптимизации. Мы сохраняем его в модель

        support_mask = self.alpha > 1e-6  # Ищем объекты, которые больше 0
        margin_mask = (self.alpha > 1e-6) & (self.alpha < self.C)     # Это уже граничные опорные векторы
        self.support_indices_ = np.where(support_mask)[0]        #self.support_indices_ будет хранить номера оп. векторов

                # Далее идет расчет ро
        if margin_mask.sum() > 0:
            rho_values = K[margin_mask] @ self.alpha       # K[margin_mask] - матрица похожести обучающих объектов между собой
            self.rho = rho_values.mean()
        else:
            rho_values = K[support_mask] @ self.alpha
            self.rho = rho_values.mean()

            return self

    def decision_function(self, X):         # Считает числовую оценку нормальности объекта
        X = np.asarray(X, dtype=float)
        K_new = self._rbf_kernel(X, self.X_train)
        scores = K_new @ self.alpha - self.rho
        return scores

    def predict(self, X):
        scores = self.decision_function(X)
        prediction = np.where(scores >= 0, 1, -1)
        return prediction
        

In [70]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
X = df.drop(columns=['attack', 'last_flag', 'true_anomaly'])
y = df['true_anomaly']

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y     # Благодаря бинарной метке соотношение будет примерно одинаковым
)
X_train_normal = X_train[y_train == 0].sample(1000, random_state=42)


cat_cols = ['protocol_type', 'service', 'flag']
num_cols = X.columns.drop(cat_cols)


preprocessor = ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)])

X_train = preprocessor.fit_transform(X_train_normal)
X_test = preprocessor.transform(X_test)

In [72]:
model = OneClassSVM(nu=0.1,
                    gamma=None,
                    max_iter=2000)
model.fit(X_train)
pred = model.predict(X_test)
pred_anomaly = pd.Series(pred, index=y_test.index).apply(lambda x: 1 if x == -1 else 0)

In [73]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score


In [74]:
print("Confusion matrix:")
print(confusion_matrix(y_test, pred_anomaly))

print("\nClassification report:")
print(classification_report(y_test, pred_anomaly))

Confusion matrix:
[[19186  1017]
 [ 1876 15713]]

Classification report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93     20203
           1       0.94      0.89      0.92     17589

    accuracy                           0.92     37792
   macro avg       0.93      0.92      0.92     37792
weighted avg       0.92      0.92      0.92     37792

